# EDA Titanic


### Introduction
First of all let's take a look at our data. Let's load it:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
base_path = Path(".")

df = pd.read_csv(base_path / 'titanic.csv', index_col='PassengerId')
print(df.shape)
df.head(10)

So we have 891 records, 11 columns (not including ID column). The `Survived` target column is our target column, the rest are measures. Here are some notes:
- we can extract from `Name` the titles like Mr, Mrs, Master, Miss (and maybe more) - this characteristic may provide some additional info
- we could probaby extract some data from ticket, but I will skip it (more appropiate for proffesional analysis)
- not everyone apparently had cabin, we need to introduce "No cabin" value
- there may be some linked features like, probably higher `pclass` will result in a higher `fare`, or extracting titles from `names` may make `sex` needless, some info might be redundant or similar.

---
### Missing values

In [ ]:
df.info()

There are some missing values as it can be seen above. We will need to do something with `Age`, `Fare`, `Embarked`, and also `Cabin` which will require special attention. The types are generally all right, age is float because of nans, it can be transformed to int with proper imputation.

---
## Exploratory Feature  analysis

#### Numerical features:

In [ ]:
df.describe()

1. 38% of passengers survived, so the target classes are imbalanced. For ML purposes, we should aim to balance the classes.
2. Most people were in 3 rd class

In [ ]:
df.groupby("Pclass")["Survived"].mean()

2. and 1st class had the highest survivability, 3rd having the worst
3. Avg age is 29 +- 14 years, quite young, here is the chart of survavibility by age groups of 10 years

In [ ]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 10, 20, 30, 40, 50, 60, 70, 80]
)

survival_by_age = df.groupby("AgeGroup", observed=True)["Survived"].mean()

sns.lineplot(
    x=survival_by_age.index.astype(str),
    y=survival_by_age.values,
    marker="o"
)

plt.xticks(rotation=45)
plt.ylabel("Survival rate")
plt.xlabel("Age")
plt.show()

3. People aged 60+ have poor survavibilty, and children were probably prioritised when it came to evacuation. And also let's look at the imputing problem, my proposition is to place it based on avg people class x sex:


In [ ]:
sns.boxplot(
    data=df,
    x="Pclass",
    y="Age",
    hue="Sex"
)
plt.show()

3. Well so we can see here that this way of imputing should be fine
4. Fare mean is 2x bigger than median of 14.5, the highest being 512

In [ ]:
sns.histplot(data=df, x="Fare")
plt.show()

4. cd. 512 is quite far away, most tickets cost below 100, also with fare imputing, same idea as with age:

In [ ]:
sns.boxplot(
    data=df,
    x="Pclass",
    y="Fare",
    hue="Sex"
)
plt.show()

5. SibSp, Parch  - over 50% of people were without spouse/siblings aboard, and 75% without children or parents, we can probably introduce a combined feature "familyMembers" 

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1 # +1 for the passenger themselves
survival_by_family = df.groupby("FamilySize")["Survived"].mean()

sns.lineplot(
    x=survival_by_family.index,
    y=survival_by_family.values,
    marker="o"
)
plt.show()

Categorical features:

In [ ]:
df.describe(include="object")

1. Let's see if titles matter when it comes to names:

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
df["Title"].value_counts()

In [ ]:
pd.crosstab(
    df["Title"],
    df["Survived"],
    normalize="index"
)

1. As seen Titles are quite important, but what I would do is change rare categories other than `Mr`, `Miss`, `Mrs` and `Master` to `Rare` 
2. 64% od passengers were men, here is survavibility by sex:

In [ ]:
sns.countplot(
    data=df,
    x="Sex",
    hue="Survived"
)
plt.show()

2. as seen, women had much better survavibility rate
3. Tickets - some people had common ones! there are only unique values. We could get data, like how many people shared the same tickets, or some info based on the ticket content, but as decided before I won't be analysing it further.
4. Cabin - not everybody had a cabin, let's see if having it benefited in any way:

In [ ]:
df["HasCabin"] = df["Cabin"].notna()
df.groupby("HasCabin")["Survived"].mean()

4. Well actually having a cabin has a huge impact, it also appears that there are shared cabins, and we can extract data from it (there appears to be sections / levels like A/B/C/D), also some people had multiple cabins separated by spaces, let's check it!

In [ ]:
df["CabinCount"] = df["Cabin"].str.split().str.len().fillna(0)

sns.countplot(
    data=df,
    x="CabinCount",
    hue="Survived"
)

df["Deck"] = df["Cabin"].str[0]

decks = sorted(df["Deck"].dropna().unique())

sns.countplot(
    data=df,
    x="Deck",
    hue="Survived",
    order=decks
)

plt.show()


4. On decks A and T, the number of passengers who died was higher than the number who survived, while the opposite was observed for all other decks. The number of cabins assigned to a passenger does not show a clear relationship with survival, suggesting that having a cabin may be more relevant than the number of cabins.
5. Embarked - only 3 missing values, easy to fill, we can see if it had any difference at all:

In [ ]:
sns.barplot(data=df, x="Embarked", y="Survived")

5. Actually there is some difference, but maybe it's related more to other characteristics like wealthness or something. ML will determine that, but it's important to remember about it, that it can be suspicious to some extent.

In [ ]:
pd.crosstab(df["Embarked"], df["Pclass"], normalize="index")

5. Yep, it is related that in the city 'C' there were 50% of passangers in the 1st class.

# Outliers

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop(["Survived"])

for col in numeric_cols:
    plt.figure()
    sns.boxplot(data=df, x=col)
    plt.title(col)
    plt.show()

categorical_cols = ["Sex", "Embarked", "Pclass", "Deck", "Title"]

for col in categorical_cols:
    counts = df[col].value_counts()

    plt.figure()
    sns.barplot(x=counts.index, y=counts.values)
    plt.title(f"Category frequency - {col}")
    plt.ylabel("Count")
    plt.show()

Except for Title, which we said we are going to replace rare categories with rare, it seems correct, there are no clear outliers, everything seems normal. We can leave 500+ outlier in Fare as it could actually be real. For some models this value can be harmful, but generally it should be alright.
Let's replace quickly rare categories in Title:

In [ ]:
common_titles = ["Mr", "Miss", "Mrs", "Master"]

df["Title"] = df["Title"].where(
    df["Title"].isin(common_titles),
    "Rare"
)

## Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop("Survived")

for col in numeric_cols:
    plt.figure()
    sns.histplot(data=df, x=col, kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()


categorical_cols = ["Deck"]

for col in categorical_cols:
    plt.figure()
    sns.countplot(
        data=df,
        x=col,
        order=sorted(df[col].dropna().unique())
    )
    plt.title(f"Distribution of {col}")
    plt.show()

Summary:
1) Pclass - oridinal distribution
2) Age - normal distribution
3) SibSp - right-skewed
4) Parch - right-skewed
5) Fare - right-skewed - we could use log-transformation - see below
6) FamilySize - right-skewed
7) Deck - normal distribution

In [ ]:
df["Fare"] = np.log1p(df["Fare"])

plt.figure()
sns.histplot(data=df, x="Fare", kde=True)
plt.title(f"Distribution of Fare")
plt.show()

Seems to be better than the original distribution

## Heatmap

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation heatmap")
plt.show()

The heatmap shows several noticeable (linear!) relationships between the numerical features. Pclass has a moderate negative correlation with Survived (-0.34), while Fare has a moderate positive correlation (0.34), which is consistent with the previous observations that passengers from higher classes and passengers who paid more were more likely to survive.

There is also a strong negative correlation between Pclass and Fare (-0.67), suggesting that passengers from higher classes generally paid higher fares. FamilySize is strongly correlated with both SibSp (0.89) and Parch (0.78), which is expected since it was directly derived from these two features.

Age has only a weak correlation with Survived (-0.08), suggesting that there is no strong linear relationship between age and survival. This does not necessarily mean that age is irrelevant, as the previous age-group analysis showed differences in survival rates.

Overall, the correlations are consistent with the previous analysis and do not reveal any unexpected relationships. The categorical features are not included in this heatmap because Pearson correlation is not appropriate for them.


## Final transformation matrix

Here is also encoding mentioned - should be self-explanatory, with Pclass - it is ordinal, since lower value = better accomodation.

And there will be no NaNs.

This is my final decision:

In [ ]:
from IPython.display import display, HTML

df_transformations = pd.read_csv("transformations.csv").replace(np.nan, "", regex=True)
display(HTML(df_transformations.to_html(index=False)))